In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
csv_path = "/content/drive/MyDrive/GEE_Exports/Table2_Annual_Landcover_Metrics_2000_2024.csv"

df = pd.read_csv(csv_path)

print(df.head())
print(df.shape)   # 应该是 (100, 8) 左右

     Year  class_id           Landcover    PET_mm_yr    AET_mm_yr       ESR  \
0  2000.0         1             Forests  1774.422865   807.468905  0.454609   
1  2000.0         2  Savanna_Shrublands  1768.197566   698.727814  0.395129   
2  2000.0         3          Grasslands  1746.799603   550.047782  0.315101   
3  2000.0         4           Croplands  1755.067692   505.527734  0.288612   
4  2001.0         1             Forests  1702.849544  1082.756679  0.634681   

   GPP_gC_m2_yr  WUE_gC_kgH2O  
0   1225.499943      1.541714  
1   1222.603404      1.762397  
2   1021.470025      1.865389  
3    938.234786      1.859886  
4   1578.451325      1.480640  
(100, 8)


In [ ]:
variables = [
    'PET_mm_yr',
    'AET_mm_yr',
    'ESR',
    'GPP_gC_m2_yr',
    'WUE_gC_kgH2O'
]

lc_order = [
    'Forests',
    'Savanna_Shrublands',
    'Grasslands',
    'Croplands'
]

summary = (
    df.groupby('Landcover')[variables]
      .agg(['mean', 'std'])
      .reindex(lc_order)
)

print(summary.round(3))

                   PET_mm_yr         AET_mm_yr             ESR         \
                        mean     std      mean      std   mean    std   
Landcover                                                               
Forests             1744.582  43.670   987.873  133.296  0.567  0.079   
Savanna_Shrublands  1710.234  59.928   822.114  112.742  0.482  0.077   
Grasslands          1684.884  82.392   628.142   98.761  0.377  0.072   
Croplands           1667.298  93.530   599.826   91.776  0.364  0.071   

                   GPP_gC_m2_yr          WUE_gC_kgH2O         
                           mean      std         mean    std  
Landcover                                                     
Forests                1476.659  199.735        1.514  0.054  
Savanna_Shrublands     1409.773  180.165        1.732  0.059  
Grasslands             1120.485  146.988        1.801  0.080  
Croplands              1054.938  131.856        1.774  0.092  


In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Friedman test
friedman_results = []

for var in variables:
    wide = df.pivot(
        index='Year',
        columns='Landcover',
        values=var
    )[lc_order].dropna()

    stat, p = friedmanchisquare(
        *[wide[lc].values for lc in lc_order]
    )

    friedman_results.append({
        'Variable': var,
        'N_years': len(wide),
        'Friedman_chi2': stat,
        'p_value': p
    })

friedman_df = pd.DataFrame(friedman_results)

print("FRIEDMAN TEST")
print(friedman_df.to_string(index=False))

FRIEDMAN TEST
    Variable  N_years  Friedman_chi2      p_value
   PET_mm_yr       25         35.544 9.349360e-08
   AET_mm_yr       25         71.832 1.729430e-15
         ESR       25         70.200 3.867513e-15
GPP_gC_m2_yr       25         75.000 3.623319e-16
WUE_gC_kgH2O       25         62.328 1.869545e-13


In [ ]:
pairwise_results = []

for var in variables:
    wide = df.pivot(
        index='Year',
        columns='Landcover',
        values=var
    )[lc_order].dropna()

    temp = []

    for g1, g2 in combinations(lc_order, 2):

        stat, p = wilcoxon(
            wide[g1],
            wide[g2],
            alternative='two-sided'
        )

        temp.append({
            'Variable': var,
            'Group1': g1,
            'Group2': g2,
            'Raw_p': p
        })

    raw_p = [x['Raw_p'] for x in temp]

    reject, p_adj, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method='holm'
    )

    for r, p_holm, sig in zip(temp, p_adj, reject):
        r['Holm_p'] = p_holm
        r['Significant'] = sig
        pairwise_results.append(r)

pairwise_df = pd.DataFrame(pairwise_results)

print(pairwise_df.to_string(index=False))

    Variable             Group1             Group2        Raw_p       Holm_p  Significant
   PET_mm_yr            Forests Savanna_Shrublands 8.803606e-05 2.641082e-04         True
   PET_mm_yr            Forests         Grasslands 1.202226e-04 2.641082e-04         True
   PET_mm_yr            Forests          Croplands 2.664328e-05 1.332164e-04         True
   PET_mm_yr Savanna_Shrublands         Grasslands 3.290772e-04 3.290772e-04         True
   PET_mm_yr Savanna_Shrublands          Croplands 4.541874e-05 1.816750e-04         True
   PET_mm_yr         Grasslands          Croplands 1.007318e-05 6.043911e-05         True
   AET_mm_yr            Forests Savanna_Shrublands 5.960464e-08 3.576279e-07         True
   AET_mm_yr            Forests         Grasslands 5.960464e-08 3.576279e-07         True
   AET_mm_yr            Forests          Croplands 5.960464e-08 3.576279e-07         True
   AET_mm_yr Savanna_Shrublands         Grasslands 5.960464e-08 3.576279e-07         True
   AET_mm_